# K-Medoids + DTW clustering of FTSE stock price patterns

**Why this method (design choices, with sources):**

- *Shape vs. time similarity.* Aghabozorgi & Teh (2014) argue that Euclidean/point-in-time
  distance treats two stocks as dissimilar if they move the same way but with a small time
  shift or lead-lag, whereas Dynamic Time Warping (DTW) allows the alignment to bend and
  correctly recognises them as similar *in shape*. This matches what the K-Means notebook
  found on this data: PCA + K-Means mostly separated stocks by trend sign (up vs down),
  not by richer pattern shape. This notebook clusters directly on DTW distance instead.

- *K-medoids instead of K-means.* Once the distance is DTW (not Euclidean), there is no
  simple "average" time series that stays meaningful (Aghabozorgi & Teh, 2014, note this
  prototype-averaging problem directly). Li, Zhu, Shen & Angelova (2022) address the same
  issue by switching from K-means to **k-medoids**, which represents each cluster by an
  *actual* member (a real stock) rather than a synthetic centroid. We do the same here.

- *Bounding DTW's cost.* Full DTW is quadratic in series length; both papers note this and
  use a bounded warping window (Sakoe-Chiba band) to keep it tractable. We do the same via
  a `WINDOW_FRAC` parameter, and this notebook also carries an optional PAA-style
  compression step (the same idea 3PTC uses for very large datasets) that is off by default
  because our datasets don't need it, but is there if you cluster something much bigger.

- *Evaluation without a Euclidean embedding.* Calinski-Harabasz and the gap statistic (used
  in the K-Means notebook) need raw Euclidean coordinates, which a DTW distance matrix
  doesn't give you. Li et al. (2022) evaluate k-medoids clustering with **Dunn Index** and
  a **Davies-Bouldin variant that works directly from a distance matrix** — both
  implemented here from their formulas, alongside silhouette (which sklearn supports
  natively on a precomputed distance matrix).

- *Stability check.* Kept in sync with the K-Means notebook: **subsampling stability**
  (Ben-Hur, Elisseeff & Guyon, 2002) — draw many 80% subsamples, cluster each, and compare
  labels pairwise via Adjusted Rand Index. This is not from either clustering paper above;
  it's a general cluster-validation technique carried over from your own K-Means notebook,
  used here purely as an admissibility filter on k (not as a ranking criterion), matching
  the update you made there.

- *Kept out of the main pipeline (future work):* Li et al. (2022) also propose a
  **Logistic-Weighted DTW (LWDTW)**, which down-weights extreme return days using a
  logistic distribution fitted to daily returns. It's a nice idea but a full pairwise
  LWDTW matrix in pure Python is too slow at this dataset's scale without a compiled
  implementation, so it's included at the very end as an optional, clearly-marked, small-
  scale ablation rather than baked into the default pipeline.

References: Aghabozorgi, S. & Teh, Y.W. (2014). *Stock market co-movement assessment using
a three-phase clustering method.* Expert Systems with Applications, 41, 1301-1314.
Li, M., Zhu, Y., Shen, Y. & Angelova, M. (2022). *Clustering-enhanced stock price prediction
using deep learning.* World Wide Web, 26, 207-232.
Ben-Hur, A., Elisseeff, A. & Guyon, I. (2002). *A stability based method for discovering
structure in clustered data.* Pacific Symposium on Biocomputing, 7, 6-17.

In [ ]:
import sys, os, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# dtaidistance gives a fast (C-backed) pairwise DTW distance matrix with a
# Sakoe-Chiba window constraint, which both source papers rely on to keep
# DTW tractable at scale.
try:
    from dtaidistance import dtw
except ImportError:
    if "google.colab" in sys.modules:
        os.system("pip install -q dtaidistance")
    else:
        os.system("pip install -q dtaidistance --break-system-packages")
    from dtaidistance import dtw

from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from sklearn.manifold import MDS

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Ready.")

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    FILENAME = list(uploaded.keys())[0]
else:
    FILENAME = "ftse250_clustering_ready.csv"
print("Using file:", FILENAME)

In [ ]:
# Identical cleaning logic to the K-Means notebook, so results stay
# directly comparable across methods for the same dataset.
MAX_MISSING_FRAC = 0.10

df = pd.read_csv(FILENAME)
date_cols = [c for c in df.columns if c != "Ticker"]

frac = df[date_cols].isna().sum(axis=1) / len(date_cols)
dropped = df.loc[frac > MAX_MISSING_FRAC, "Ticker"].tolist()
df = df.loc[frac <= MAX_MISSING_FRAC].reset_index(drop=True)

n_filled = int(df[date_cols].isna().sum().sum())
df[date_cols] = df[date_cols].ffill(axis=1).bfill(axis=1)

tickers = df["Ticker"].values
A = df[date_cols].values.astype(float)
dates = pd.to_datetime(date_cols)

print(f"Dropped (>10% missing) : {len(dropped)} {dropped}")
print(f"Single gaps filled     : {n_filled}")
print(f"Stocks retained        : {len(df)}")
print(f"Trading days            : {len(date_cols)}")
print(f"Date range              : {date_cols[0]} to {date_cols[-1]}")
assert df[date_cols].isna().sum().sum() == 0

In [ ]:
# Optional: PAA-style block-average compression before computing DTW.
# This is the same idea 3PTC (Aghabozorgi & Teh, 2014) uses to keep very
# large time-series datasets tractable in its first phase. Our three FTSE
# files (up to ~2,800 trading days) are small enough that full-resolution
# windowed DTW already runs in well under a minute, so this is OFF by
# default. Flip COMPRESS_FOR_DTW to True if you ever cluster something with
# many thousands of time points and DTW becomes too slow.

COMPRESS_FOR_DTW = False
TARGET_POINTS = 250

def paa_compress(A, target_points):
    n, T = A.shape
    if T <= target_points:
        return A
    bounds = np.linspace(0, T, target_points + 1).astype(int)
    return np.column_stack([
        A[:, bounds[i]:max(bounds[i] + 1, bounds[i + 1])].mean(axis=1)
        for i in range(target_points)
    ])

A_dtw = paa_compress(A, TARGET_POINTS) if COMPRESS_FOR_DTW else A
print(f"Series length used for DTW: {A_dtw.shape[1]}" +
      ("  (PAA-compressed)" if COMPRESS_FOR_DTW else "  (full resolution)"))

In [ ]:
# Look at 5 random stocks
idx = np.random.default_rng(RANDOM_STATE).choice(len(tickers), 5, replace=False)
fig, ax = plt.subplots(figsize=(12, 4.5))
for i in idx:
    ax.plot(dates, A[i], linewidth=1, label=tickers[i])
ax.set_title("Sample of standardised price series")
ax.set_xlabel("Date"); ax.set_ylabel("Standardised price (z-score)")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Pairwise DTW distance matrix, bounded by a Sakoe-Chiba window (both papers
# use this style of banding to keep DTW's O(T^2)-per-pair cost manageable).
# WINDOW_FRAC=0.05 means warps of up to ~5% of the series length are allowed.
WINDOW_FRAC = 0.05
MIN_WINDOW = 10

T = A_dtw.shape[1]
WINDOW = max(MIN_WINDOW, int(WINDOW_FRAC * T))

t0 = time.time()
D = dtw.distance_matrix_fast(A_dtw, window=WINDOW)
D = np.maximum(D, D.T)      # defensive symmetrise regardless of dtaidistance's fill convention
np.fill_diagonal(D, 0.0)
print(f"DTW distance matrix computed in {time.time()-t0:.1f}s")
print(f"Series length: {T}, window: {WINDOW} days")
print(f"Distance matrix shape: {D.shape}")
print(f"Distance range: [{D.min():.3f}, {D.max():.3f}], mean {D[D>0].mean():.3f}")

In [ ]:
# ---- K-medoids (PAM-style: seed with k-medoids++, then alternate
#      assign -> move each medoid to the in-cluster point minimising total
#      in-cluster distance -> repeat). Works directly off a precomputed
#      distance matrix, which is what k-medoids needs and k-means can't use.

def _kmedoids_plusplus_init(D, k, rng):
    n = D.shape[0]
    medoids = [int(rng.integers(n))]
    for _ in range(1, k):
        dist_to_nearest = D[:, medoids].min(axis=1)
        probs = dist_to_nearest ** 2
        s = probs.sum()
        probs = probs / s if s > 0 else np.ones(n) / n
        medoids.append(int(rng.choice(n, p=probs)))
    return np.array(medoids)


def kmedoids_pam(D, k, n_init=10, max_iter=100, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    n = D.shape[0]
    best_cost, best_medoids, best_labels = np.inf, None, None
    for init in range(n_init):
        medoids = _kmedoids_plusplus_init(D, k, rng)
        for _ in range(max_iter):
            labels = np.argmin(D[:, medoids], axis=1)
            new_medoids = medoids.copy()
            changed = False
            for ci in range(k):
                members = np.where(labels == ci)[0]
                if len(members) == 0:
                    continue
                costs = D[np.ix_(members, members)].sum(axis=1)
                best_local = members[np.argmin(costs)]
                if best_local != new_medoids[ci]:
                    new_medoids[ci] = best_local
                    changed = True
            if not changed:
                break
            medoids = new_medoids
        labels = np.argmin(D[:, medoids], axis=1)
        cost = D[np.arange(n), medoids[labels]].sum()
        if cost < best_cost:
            best_cost, best_medoids, best_labels = cost, medoids.copy(), labels.copy()
    return best_medoids, best_labels, best_cost


def dunn_index(D, labels):
    """Min inter-cluster distance / max intra-cluster distance (Li et al., 2022, Eq. 12).
    Higher is better."""
    clusters = np.unique(labels)
    intra_max = 0.0
    for c in clusters:
        idx = np.where(labels == c)[0]
        if len(idx) > 1:
            intra_max = max(intra_max, D[np.ix_(idx, idx)].max())
    inter_min = np.inf
    for i, ci in enumerate(clusters):
        for cj in clusters[i + 1:]:
            idx_i = np.where(labels == ci)[0]
            idx_j = np.where(labels == cj)[0]
            inter_min = min(inter_min, D[np.ix_(idx_i, idx_j)].min())
    return inter_min / intra_max if intra_max > 0 else np.nan


def davies_bouldin_distmat(D, labels, medoids):
    """Davies-Bouldin adapted to work from a precomputed distance matrix
    (Li et al., 2022, Eq. 13), using each medoid as its cluster's centre and
    mean in-cluster distance to the medoid as dispersion. Lower is better."""
    k = len(medoids)
    S = np.array([D[np.where(labels == ci)[0], medoids[ci]].mean() for ci in range(k)])
    total = 0.0
    for ci in range(k):
        best = 0.0
        for cj in range(k):
            if ci == cj:
                continue
            d_ij = D[medoids[ci], medoids[cj]]
            ratio = (S[ci] + S[cj]) / d_ij if d_ij > 0 else np.inf
            best = max(best, ratio)
        total += best
    return total / k


def subsample_stability_dm(D, k, B=100, frac=0.8, n_init=3, min_overlap=10, seed=RANDOM_STATE):
    """Subsampling stability (Ben-Hur et al., 2002), adapted to a
    precomputed distance matrix and k-medoids -- same technique and
    parameterisation (B, frac, floor) as the current K-Means notebook, kept
    in sync so the two methods are validated the same way.

    Draws B subsamples of size frac*n WITHOUT replacement, clusters each
    with k-medoids on the corresponding distance sub-matrix, and compares
    every pair of runs with ARI on the points they share. Returns
    (mean ARI, min ARI).
    """
    rng = np.random.default_rng(seed)
    n = D.shape[0]
    m = int(frac * n)

    # labels[b, p] = cluster of point p in run b, or -1 if p was not sampled
    labels = np.full((B, n), -1, dtype=int)
    for b in range(B):
        idx = rng.choice(n, m, replace=False)
        sub = D[np.ix_(idx, idx)]
        _, lab, _ = kmedoids_pam(sub, k, n_init=n_init, random_state=seed + b)
        labels[b, idx] = lab

    aris = []
    for i in range(B):
        for j in range(i + 1, B):
            common = (labels[i] >= 0) & (labels[j] >= 0)
            if common.sum() < min_overlap:
                continue
            aris.append(adjusted_rand_score(labels[i, common], labels[j, common]))

    if not aris:
        return np.nan, np.nan
    return float(np.mean(aris)), float(np.min(aris))


print("Functions defined.")

In [ ]:
K_RANGE = range(2, 11)
N_SUBSAMPLES = 100
SUBSAMPLE_FRAC = 0.8
STABILITY_FLOOR = 0.65

t0 = time.time()
n = D.shape[0]
rows = []
for k in K_RANGE:
    medoids_k, labels_k, cost_k = kmedoids_pam(D, k, n_init=10)
    stab, stab_min = subsample_stability_dm(D, k, B=N_SUBSAMPLES, frac=SUBSAMPLE_FRAC)
    smallest = int(np.bincount(labels_k, minlength=k).min())
    rows.append({
        "k": k,
        # sample_size caps the O(n^2) silhouette cost on large datasets
        "silhouette": silhouette_score(D, labels_k, metric="precomputed",
                                       sample_size=min(n, 5000), random_state=RANDOM_STATE),
        "dunn": dunn_index(D, labels_k),
        "davies_bouldin": davies_bouldin_distmat(D, labels_k, medoids_k),
        "stability": stab,
        "stability_min": stab_min,
        "smallest_cluster": smallest,
        "smallest_pct": 100 * smallest / n,
    })

metrics = pd.DataFrame(rows)
metrics["stable"] = metrics["stability"] >= STABILITY_FLOOR   # apply the floor
print(f"Computed in {time.time()-t0:.0f}s\n")
display(metrics.round(4))

In [ ]:
# Stability is the admissibility filter, so it is NOT also used as a ranking criterion
# (matches the current K-Means notebook's approach).
# value = True means "lower is better"
RANK_METRICS = {
    "silhouette": False,
    "dunn": False,
    "davies_bouldin": True,
}

passes = metrics["stability"] >= STABILITY_FLOOR      # NaN -> False
floor_met = bool(passes.any())
if not floor_met:
    print(f"WARNING: no k reaches the stability floor of {STABILITY_FLOOR}.")
    print("Clusters are not reproducible under 80% subsampling at any k, so treat")
    print("the result below as tentative (or revisit window size/features/algorithm).\n")
    passes = pd.Series(True, index=metrics.index)

# Rank only among admissible k, so ranks are comparable with each other
adm = metrics[passes].copy()
for col, lower_is_better in RANK_METRICS.items():
    adm[f"r_{col}"] = adm[col].rank(ascending=lower_is_better)
adm["mean_rank"] = adm[[f"r_{c}" for c in RANK_METRICS]].mean(axis=1)

# Ties on mean_rank go to the smaller k (parsimony)
best = adm.sort_values(["mean_rank", "k"]).iloc[0]
consensus_k = int(best["k"])

# Keep ranks visible in the main table (NaN for inadmissible k)
metrics = metrics.drop(columns=[c for c in metrics.columns
                                if c.startswith("r_") or c == "mean_rank"])
metrics = metrics.join(adm[[f"r_{c}" for c in RANK_METRICS] + ["mean_rank"]])

print(f"Silhouette alone would pick k = {int(metrics.loc[metrics['silhouette'].idxmax(), 'k'])}")
print(f"Dunn Index alone would pick   k = {int(metrics.loc[metrics['dunn'].idxmax(), 'k'])}")
print(f"Consensus of {len(RANK_METRICS)} metrics among "
      f"{'stable' if floor_met else 'ALL (none stable)'} k   k = {consensus_k}")
print(f"\nStability at consensus k    : {best['stability']:.3f}")
print(f"Smallest cluster at consensus k: {int(best['smallest_cluster'])} "
      f"({best['smallest_pct']:.1f}%)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
specs = [("silhouette", "Silhouette (higher better)"),
         ("dunn", "Dunn Index (higher better)"),
         ("davies_bouldin", "Davies-Bouldin, distance-matrix form (lower better)"),
         ("stability", "Subsampling stability, mean ARI (higher better)")]

ks = metrics["k"]
fails = ~(metrics["stability"] >= STABILITY_FLOOR)     # NaN counts as failing
floor_met = bool((~fails).any())

for ax, (col, title) in zip(axes.flat, specs):
    ax.plot(ks, metrics[col], marker="o", color="C0", zorder=2)
    # hollow grey markers = k values that fail the stability floor
    ax.scatter(ks[fails], metrics.loc[fails, col], s=60,
               facecolors="white", edgecolors="grey", zorder=3)
    ax.axvline(consensus_k, color="green", ls=":", lw=1.5)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("k")
    ax.set_xticks(ks)

    if col == "stability":
        ax.plot(ks, metrics["stability_min"], marker=".", ls="--",
                color="C1", label="min pairwise ARI")
        ax.axhline(STABILITY_FLOOR, color="red", ls="--", lw=1, label="floor")
        ax.legend(fontsize=8, loc="best")

fig.suptitle(f"Consensus k = {consensus_k}"
             + ("" if floor_met else "  (WARNING: no k passes the stability floor)"),
             fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
FINAL_K = consensus_k      # override manually if you prefer a different k

assert len(tickers) == D.shape[0], "tickers and D are misaligned"

medoids, labels, cost = kmedoids_pam(D, FINAL_K, n_init=20)
stab, stab_min = subsample_stability_dm(D, FINAL_K, B=N_SUBSAMPLES, frac=SUBSAMPLE_FRAC)
sizes = np.bincount(labels, minlength=FINAL_K)
floor_ok = stab >= STABILITY_FLOOR                        # NaN -> False

print("=" * 50)
print(f"  DATASET     : {FILENAME}")
print(f"  Stocks      : {len(tickers)}")
print(f"  Distance    : DTW, window={WINDOW} ({WINDOW_FRAC:.0%} of series length)")
print(f"  k           : {FINAL_K}")
print("-" * 50)
print(f"  Silhouette     : {silhouette_score(D, labels, metric='precomputed'):.4f}")
print(f"  Dunn Index     : {dunn_index(D, labels):.4f}")
print(f"  Davies-Bouldin : {davies_bouldin_distmat(D, labels, medoids):.4f}")
print(f"  Stability      : {stab:.4f}  (worst pair {stab_min:.3f})")
print(f"  Stability floor: {STABILITY_FLOOR}  ->  {'PASS' if floor_ok else 'FAIL'}")
print("-" * 50)
print(f"  Cluster sizes  : {sizes.tolist()}")
print(f"  Sizes (%)      : {[round(100 * s / len(labels), 1) for s in sizes]}")
print(f"  Medoid tickers : {list(tickers[medoids])}")
print("=" * 50)

# Ticker -> cluster table for downstream use
assignments = pd.DataFrame({"ticker": list(tickers), "cluster": labels})

In [ ]:
# 2D layout via MDS on the DTW distance matrix itself (a PCA scatter would
# be a Euclidean-space view of a non-Euclidean distance, so MDS is the
# honest equivalent here). Medoids -- the real representative stocks -- are
# starred.
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=RANDOM_STATE,
          normalized_stress=False)
V = mds.fit_transform(D)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(V[:, 0], V[:, 1], c=labels, cmap="tab10", s=45, edgecolor="k", linewidth=0.3)
ax.scatter(V[medoids, 0], V[medoids, 1], marker="*", s=260, c="black", label="medoids")
for m in medoids:
    ax.annotate(tickers[m], (V[m, 0], V[m, 1]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_title(f"K-Medoids + DTW clusters (k={FINAL_K}) — {FILENAME}")
ax.set_xlabel("MDS dim 1"); ax.set_ylabel("MDS dim 2")
ax.add_artist(ax.legend(*sc.legend_elements(), title="Cluster", loc="upper left"))
plt.tight_layout(); plt.show()

In [ ]:
n_cols = 2
n_rows = int(np.ceil(FINAL_K / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.6 * n_rows), squeeze=False)

for c in range(FINAL_K):
    ax = axes[c // n_cols][c % n_cols]
    mem = A[labels == c]
    mu, sd = mem.mean(axis=0), mem.std(axis=0)
    ax.plot(dates, mu, color=f"C{c}", lw=1.4, label="cluster mean")
    ax.fill_between(dates, mu - sd, mu + sd, color=f"C{c}", alpha=0.2)
    ax.plot(dates, A[medoids[c]], color="black", lw=1.3, ls="--", label=f"medoid ({tickers[medoids[c]]})")
    ax.set_title(f"Cluster {c}  (n={len(mem)})")
    ax.set_ylabel("Standardised price")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.legend(fontsize=8)

for c in range(FINAL_K, n_rows * n_cols):
    axes[c // n_cols][c % n_cols].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
sv = silhouette_samples(D, labels, metric="precomputed")
fig, ax = plt.subplots(figsize=(9, 6))
y = 10
for c in range(FINAL_K):
    v = np.sort(sv[labels == c])
    ax.fill_betweenx(np.arange(y, y + len(v)), 0, v, facecolor=f"C{c}", alpha=0.7)
    ax.text(-0.02, y + 0.5 * len(v), str(c))
    y += len(v) + 10
ax.axvline(sv.mean(), color="red", ls="--", label=f"mean = {sv.mean():.3f}")
ax.set_title("Silhouette plot (precomputed DTW distance)"); ax.set_xlabel("Silhouette coefficient")
ax.set_yticks([]); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
import textwrap
for c in range(FINAL_K):
    mem = sorted(tickers[labels == c])
    print(f"Cluster {c} (n={len(mem)}, medoid={tickers[medoids[c]]}):")
    labelled = [f"*{t}*" if t == tickers[medoids[c]] else t for t in mem]
    print(textwrap.fill(", ".join(labelled), width=95))
    print()

In [ ]:
# Same honesty check as the K-Means notebook: is this clustering just a
# restatement of trend direction, or is it picking up something beyond that?
t_idx = np.arange(A.shape[1])
slopes = np.array([np.polyfit(t_idx, row, 1)[0] for row in A])
trend_sign = (slopes > 0).astype(int)

ari_trend = adjusted_rand_score(labels, trend_sign)
print(f"ARI between clusters and sign-of-trend: {ari_trend:.3f}")
print()
if ari_trend > 0.7:
    print("=> The clustering is largely a restatement of trend direction.")
elif ari_trend > 0.3:
    print("=> Trend explains part of the clustering, but not all of it.")
else:
    print("=> The clustering is capturing something beyond trend direction.")

print(f"\nStocks trending up: {trend_sign.sum()} / down: {(1-trend_sign).sum()}")
print("\nMean slope per cluster:")
for c in range(FINAL_K):
    print(f"  Cluster {c}: {slopes[labels == c].mean():+.6f}")

## Optional / advanced: Logistic-Weighted DTW (LWDTW) ablation

Li et al. (2022) fit a logistic distribution to daily *returns* and use its density as a
weight on the DTW cost function, so ordinary-sized daily moves count more than extreme
outlier days when judging similarity. This cell implements that idea faithfully to their
Eqs. 3 and 8-11, but **only as a pure-Python pairwise function for small-scale checks** —
computing a full N x N LWDTW matrix this way (no C/numba backend) would be far slower than
the `dtaidistance`-backed cell above at this dataset's size. Treat this as something to
try on a handful of stocks, or as a documented direction for future work, not as a
drop-in replacement for the main pipeline.

In [ ]:
def logistic_weight(r, mu, s):
    z = (r - mu) / s
    return np.exp(-z) / (s * (1 + np.exp(-z)) ** 2)

def lwdtw_distance(x, y, window=None):
    """Pairwise Logistic-Weighted DTW between two z-scored price series
    (Li et al., 2022). Returns are approximated as first differences of the
    standardised price series, since that's what these CSVs give us
    (the paper works from raw daily log returns)."""
    n, m = len(x), len(y)
    window = window or max(n, m)
    rx = np.diff(x, prepend=x[0])
    ry = np.diff(y, prepend=y[0])
    all_r = np.concatenate([rx, ry])
    mu, s = all_r.mean(), all_r.std() * np.sqrt(3) / np.pi  # logistic-scale approx from std

    INF = np.inf
    Dm = np.full((n + 1, m + 1), INF)
    Dm[0, 0] = 0.0
    for i in range(1, n + 1):
        lo, hi = max(1, i - window), min(m, i + window)
        for j in range(lo, hi + 1):
            cost = (x[i - 1] - y[j - 1]) ** 2
            w = logistic_weight(0.5 * (rx[i - 1] + ry[j - 1]), mu, s)
            Dm[i, j] = w * cost + min(Dm[i - 1, j], Dm[i, j - 1], Dm[i - 1, j - 1])
    return np.sqrt(Dm[n, m])

# Example: try it on a small handful of stocks rather than the full dataset.
# sample_idx = np.random.default_rng(RANDOM_STATE).choice(len(tickers), 15, replace=False)
# small_D = np.zeros((len(sample_idx), len(sample_idx)))
# for a, i in enumerate(sample_idx):
#     for b, j in enumerate(sample_idx):
#         if b > a:
#             d = lwdtw_distance(A[i], A[j], window=WINDOW)
#             small_D[a, b] = small_D[b, a] = d
# print(small_D.round(2))